# UniProt RAG — query & summarize

Pipeline: **ingest** → **index** → **search** → **summarize** (search and summarize run in this notebook).

Prerequisites: `python src/ingest_uniprot.py` then `python src/build_uniprot_index.py`. EDA: `uniprot_eda.ipynb`.

In [ ]:
# Optional: install dependencies if needed
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [ ]:
# Natural-language query
query = "mismatch repair mechanism"

In [ ]:
# Retrieval limits
TOP_K_CHUNKS = 20  # chunk pool from Chroma; used to rank proteins (not sent to LLM)
TOP_K_RESULTS = 3  # top proteins by best-chunk distance; LLM gets 1 full parquet row each

In [ ]:
# Imports
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer


In [ ]:
# Project paths and shared RAG helpers
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from rag_search import search
from rag_summary import load_llm, summarize


In [ ]:
# Paths, embedding model, search config, LLM prompt
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
UNIPROT_PARQUET = data_processed / "uniprot_rag.parquet"
UNIPROT_COLLECTION = "protein_chunks"
UNIPROT_RECORD_FIELDS = [
    "Entry", "Entry Name", "Gene Names", "Protein names", "Length",
    "Function [CC]", "Involvement in disease", "Subcellular location [CC]",
    "Domain [FT]", "Region", "Zinc finger", "Domain [CC]",
    "Natural variant", "Polymorphism", "Tissue specificity",
    "Developmental stage", "Interacts with",
]

# Maps Chroma chunk metadata and parquet columns to rag_search.search()
SEARCH_CONFIG = {
    "collection_name": UNIPROT_COLLECTION,  # Chroma collection to query
    "parquet_path": UNIPROT_PARQUET,        # full records loaded for top-ranked proteins
    "group_key": "entry_name",              # metadata field: collapse chunks to one protein
    "record_id_column": "Entry",            # parquet column used to look up full records
    "record_id_meta_key": "accession",      # chunk metadata field holding the same ID
    "record_fields": UNIPROT_RECORD_FIELDS, # parquet columns included in LLM context
    "section_header_template": "### Result {rank} — entry_name={entry_name}, accession={accession}, gene={gene}",  # per-result header; placeholders from metadata
    "hit_fields": [("entry_name", "entry_name"), ("accession", "accession"), ("gene", "gene")],  # metadata keys printed for each hit
    "full_record_label": "Full UniProt record",       # label above the joined parquet row
    "selected_label": "protein record(s) for context",  # verbose log when loading records
}

# Needs ~8GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# Needs ~4GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = (
    "You are a protein biology assistant. Summarize UniProt protein search results "
    "for the user's query using ONLY the full records provided. "
    "Be concise, accurate, and structured. For each relevant protein, mention entry name, "
    "accession, gene, function, disease involvement, and tissue expression when available. "
    "If information is missing, say so. Do not invent facts."
)


## Search

**Ranking:** Chroma returns `TOP_K_CHUNKS` chunks. Each protein keeps its closest chunk as its score; the top `TOP_K_RESULTS` proteins by that score are selected. The LLM receives one full parquet row per protein (no chunk text).

Printed sections:

1. **RETRIEVAL** — chunk pool size and how top proteins are picked
2. **TOP N RECORD(S)** — ranked proteins with best-chunk preview (debug only)
3. **LLM CONTEXT** — full records sent to `summarize()` (preview may be truncated)

The tqdm bar is embedding model loading, not search output.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL)
# returns ranked hits + context string for the LLM
result = search(
    query,
    SEARCH_CONFIG,
    top_k_chunks=TOP_K_CHUNKS,
    top_k_results=TOP_K_RESULTS,
    chroma_dir=chroma_dir,
    embed_model=embed_model,
)

## Summarize

Load LLM (once per kernel session) → generate answer from retrieved context only.

- **Download** once → cached under `~/.cache/huggingface/`.
- **RAM:** 0.5B ~2GB · re-run this cell reuses the loaded model.


LLM will only receive uniprot records for the top 3 proteins, not the chunks from top 20, to make it computationally easier, compatible with the light model 0.5B

In [ ]:
# load_llm reuses llm_model/llm_tokenizer if already in RAM
llm_model, llm_tokenizer = load_llm(
    LLM_MODEL,
    model=globals().get("llm_model"),
    tokenizer=globals().get("llm_tokenizer"),
)


In [ ]:
summary = summarize(
    query,
    result["context"],
    llm_model,
    llm_tokenizer,
    SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
)
print("=" * 60)
print(f"Query: {query}\n")
print(summary)



The query "mismatch repair mechanism" was answered using the following evidence:

UniProt entry: P52701
Accession: P52701
Gene: MSH6
Description: Component of the post-replicative DNA mismatch repair system (MMR). Heterodimerizes with MSH2 to form MutS alpha, which binds to DNA mismatches thereby initiating DNA repair.

This entry describes the MSH6 protein, which plays a key role in the post-replicative DNA mismatch repair system. Specifically, it mentions that MSH6 heterodimerizes with MSH2 to form MutS alpha, which binds to DNA mismatches to initiate DNA repair.


The query "mismatch repair mechanism" was answered using the following UniProt protein searches:

UniProt ID: P52701
Protein: MSH6_HUMAN
Accession: P52701
Gene: MSH6
Function: DNA mismatch repair protein Msh6 (hMSH6) (G/T mismatch-binding protein) (GTBP) (GTMBP) (MutS protein homolog 6) (MutS-alpha 160 kDa subunit) (p160)

This indicates that MSH6 is involved in mismatch repair mechanisms, particularly DNA mismatch repair proteins like Msh6.

In [ ]:
import re
re.findall(r"accession=([A-Z0-9]+)", result["context"])
# ['P52701', 'P43246', 'P20585']

# retrieval correctly sent MSH6, MSH2, and MSH3; the answer only discussed P52701 because the tiny LLM didn’t use the full context well.